# News Intelligence Pipeline

## PROJECT GOAL
Build an end-to-end NLP pipeline that ingests real-world news, automatically summarises articles, performs sentiment analysis, extracts key entities, and delivers context-aware answers using a RAG-based approach.

**`Problem Statement`** <br>
Large volumes of unstructured news articles are published daily across diverse topics, making it difficult to efficiently extract key information and understand context. This project builds an end-to-end pipeline to process real-time news data and transform raw text into structured, meaningful intelligence.


**Input**: Real-time news articles ingested from **NewsAPI** across a wide range of topics. <br>
**Output**: Structured insights including article summaries, sentiment scores, extracted named entities (locations, organisations, persons), and context-aware answers generated via a RAG pipeline using a local Mistral model.

## Import Libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Import core python libraries
import os
import sys
import hashlib
import datetime
import numpy as np
from pathlib import Path

# Import DS libraries
import torch
import pandas as pd
import matplotlib.pyplot as plt
from transformers import pipeline
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

D:\Anaconda\envs\disasterNLP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Import Custom Modules
from scripts.utils.env_loader import get_env_var
from scripts.utils.model_loader import load_models
from scripts.utils.config_loader import load_config
from scripts.nlp.documents import prepare_documents
from scripts.pipeline.news_pipeline import run_news_pipeline
from scripts.nlp.query_handler import extract_query_entities
from scripts.nlp.context import context_forming, generate_answer
from scripts.nlp.retriever import filter_by_metadata, similarity_search
from scripts.nlp.embeddings import generate_embeddings, build_embeddings
from scripts.nlp.vector_store import load_vector_store, update_vector_store
from scripts.preprocessing.article_preprocessing import drop_post_nlp_columns

## Load Models & Parameters

Load the NewsAPI key from the `.env` file. This key is required to fetch live news articles.

In [4]:
news_api_key = get_env_var("NEWS_API_KEY")

Load the project configuration and extract default parameters such as paths, model settings, and API endpoints into a flat dictionary for easy access throughout the pipeline.

In [5]:
config = load_config()
default_parameters = {
    "raw_data_path": config["paths"]["raw_data"],
    "logs_path": config["paths"]["logs"],
    "vector_store_path": config["paths"]["vector_store"],
    "max_text_length": config["defaults"]["max_text_length"],
    "min_text_length": config["defaults"]["min_text_length"],
    "max_top_k_results": config["defaults"]["top_k_results"],
    "ep_everything": config["end_points"]["everything"],
    "ep_top_headlines": config["end_points"]["top-headline"],
    "ep_top_headline_sources": config["end_points"]["top-headline-sources"]
}

Load all ML models into memory: sentiment analysis (RoBERTa), NER, summarization, KeyBERT for keyword extraction, and the sentence transformer for embeddings.

> **Note:** First load takes ~20-30 seconds depending on hardware. Subsequent runs are instant.

In [6]:
model_pipelines = load_models(config)

Device set to use cuda:0
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0
Device set to use cuda:0


Inspect the loaded model pipelines to confirm all models initialised correctly.

In [7]:
model_pipelines

{'device': 0,
 'ner_pipeline': <transformers.pipelines.token_classification.TokenClassificationPipeline at 0x22928807e10>,
 'summarizer_tokenizer': BartTokenizerFast(name_or_path='facebook/bart-large-cnn', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
 	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
 	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
 	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
 	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
 	50264: AddedToken("<mask

---
## Case Study: "What's happening with the U.S. economy?"

This notebook walks through the full pipeline end-to-end using a single real-world query. Each step mirrors exactly what happens inside the Streamlit app when a user submits a search.

Dates are fixed for reproducibility of this case study.

## User Query

In [8]:
# Start Time
start = datetime.datetime.now()

In [9]:
user_query = "What's happening with the U.S. economy?"

to_date = datetime.date(2026, 3, 22)
from_date = datetime.date(2026, 3, 15)

### Step 1: Embed the query

Convert the user query into a dense vector using the sentence transformer model. This vector is used to perform similarity search against the FAISS index.

In [10]:
user_query_vector = generate_embeddings(
    model_pipelines['embedding_model'],
    [user_query]
)[0]

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.44it/s]


### Step 2: Extract entities and keywords

Run NER to extract named entities (persons, locations, organisations) from the query. Simultaneously, KeyBERT extracts semantically important keywords.

Both are combined to form a rich search keyword for NewsAPI. For example, `"U.S. AND economy"` and to build query metadata used for filtering the vector store.

In [11]:
query_entities, extracted_entities, keyword = extract_query_entities(
    user_query, model_pipelines.get('ner_pipeline'), model_pipelines.get('keybert_model')
)

print(f"Query Entities: {query_entities}")
print(f"Keyword: {keyword}")

Query Entities: {'persons': [], 'locations': ['U.S'], 'organizations': []}
Keyword: U.S AND economy AND happening


D:\Anaconda\envs\disasterNLP\Lib\site-packages\transformers\pipelines\token_classification.py:490: UserWarning: Tokenizer does not support real words, using fallback heuristic
  warnings.warn(


Build the query metadata dictionary, entities and date range are used to filter retrieved documents from the vector store.

In [12]:
query_metadata = {
    'severity': None,
    'sentiment_label': None,
    'persons': query_entities.get('persons'),
    'locations': query_entities.get('locations'),
    'organizations': query_entities.get('organizations'),
    'from_date': from_date,
    'to_date': to_date
}

### Step 3: Search the existing vector store

Load the persisted FAISS index from disk and perform similarity search using the query vector. If relevant documents are found (HIT), they are used directly. Otherwise the pipeline fetches fresh articles (MISS).

In [13]:
# Load FAISS index
faiss_index, metadata_store = load_vector_store(default_parameters["vector_store_path"])
nearest_k_indices = np.array([])
if faiss_index is not None and metadata_store is not None:
    _, nearest_k_indices = similarity_search(
        faiss_index,
        user_query_vector,
        k = default_parameters["max_top_k_results"]
    )

Filter the nearest neighbour indices using query metadata (i.e. date range, locations, persons, and organisations) to ensure contextual relevance.

In [14]:
# Filter indices using query metadata
filtered_indices = filter_by_metadata(
    indices = nearest_k_indices,
    metadata_store = metadata_store or [],
    filter_criteria = query_metadata
)

## Filtered Indices from the current vector store
print(f"Filtered Indices: {filtered_indices}")

Filtered Indices: [164, 156, 179, 167, 24, 160, 175]


Retrieve document content and metadata for the filtered indices. These are the candidate documents from the existing vector store.

In [15]:
# Filtered Documents
documents = [
    {"page_content": metadata_store[i].get("page_content"), "metadata": metadata_store[i]}
    for i in filtered_indices
]

## Ingest News

### Step 4: Fetch fresh news articles

Fetch new articles from NewsAPI using the extracted keyword. The full NLP enrichment pipeline (preprocessing, sentiment analysis, and NER) runs on every fetched article before storage.

In [16]:
news_query_params = {
    'end_point': default_parameters['ep_everything'],
    'q': keyword,
    'search_in': None,
    'source': None,
    'domain': None,
    'excluded_domains': None,
    'from_date': str(from_date),
    'to_date': str(to_date),
    'language': 'en',
    'sort_by': 'publishedAt'
}

device = 0
device = device if (device >= 0 and torch.cuda.is_available()) else -1
print(f"Device: {device}")

Device: 0


Run the full ingestion pipeline (fetch articles, preprocess text, run sentiment analysis, NER, and optionally summarization). All enrichment is stored as metadata alongside each article.

In [17]:
article_df = run_news_pipeline(
    query_params = news_query_params,
    api_key = news_api_key,
    log_file_path = default_parameters.get('logs_path'),
    raw_data_path = default_parameters.get('raw_data_path'),
    sentiment_pipeline = model_pipelines.get('sentiment_pipeline'),
    summarizer_tokenizer = model_pipelines.get('summarizer_tokenizer'),
    summarization_pipeline = model_pipelines.get('summarizer_pipeline'),
    ner_pipeline = model_pipelines.get('ner_pipeline'),
    default_min_text_length = default_parameters.get('min_text_length', 30),
    default_max_text_length = default_parameters.get('max_text_length', 150),
    summarize = False
)

D:\Anaconda\envs\disasterNLP\Lib\site-packages\transformers\pipelines\token_classification.py:490: UserWarning: Tokenizer does not support real words, using fallback heuristic
  warnings.warn(


Preview the ingested articles with all NLP enrichment columns.

In [18]:
article_df.head()

,title,url,source_name,news_content,published_date,article_id,summary,summary_token_count,summary_word_count,sentiment_label,sentiment_score,severity,persons,locations,organizations
0,Apocalypse Rising: We Have Reached A Moment In...,http://theeconomiccollapseblog.com/apocalypse-...,Theeconomiccollapseblog.com,Apocalypse Rising: We Have Reached A Moment In...,2026-03-22,3c660bfc2f1ad456a114cc3cb1682ef2,Apocalypse Rising: We Have Reached A Moment In...,70,58,Negative,0.530233,High,[Trump],[Iran],[Rising]
1,"Full transcript of ""Face the Nation with Marga...",https://www.cbsnews.com/news/face-the-nation-f...,CBS News,"Full transcript of ""Face the Nation with Marga...",2026-03-22,7943de934befea3833290627ab3472c3,"Full transcript of ""Face the Nation with Marga...",47,34,Neutral,0.953054,Medium,"[Mike Waltz, Mark Rutte, Margaret Brennan]",[],"[NATO, U.N]"
2,Transcript: NATO Secretary General Mark Rutte ...,https://www.cbsnews.com/news/mark-rutte-nato-s...,CBS News,Transcript: NATO Secretary General Mark Rutte ...,2026-03-22,cded56559172cb24b5c01d2bd98ea045,Transcript: NATO Secretary General Mark Rutte ...,56,43,Neutral,0.958559,Medium,"[Margaret Brennan, Mark Rutte]",[],[NATO]
3,"Sunday Summary: War, Oil Blockades, Interest R...",http://commercialobserver.com/2026/03/sunday-s...,Commercial Observer,"Sunday Summary: War, Oil Blockades, Interest R...",2026-03-22,e5e727303ddb3e511d9f95d0a2dfff95,"Sunday Summary: War, Oil Blockades, Interest R...",69,58,Negative,0.593892,High,[],"[Israel, United States, Iran]",[]
4,Bridging medical realities in the study of tec...,https://news.mit.edu/2026/bridging-medical-rea...,Mit.edu,Bridging medical realities in the study of tec...,2026-03-22,edbeb8c554de6c5b3aa19d86ed858aa8,Bridging medical realities in the study of tec...,59,50,Neutral,0.795128,Medium,[Amy Moran-Thomas],[],[MIT]


Drop intermediate NLP processing columns not needed for embedding or retrieval.

In [19]:
article_df = drop_post_nlp_columns(article_df)

# Number of articles fetched
print(f"Number of articles fetched: {len(article_df)}")

Number of articles fetched: 32


Preview the cleaned dataframe ready for the RAG pipeline.

In [20]:
article_df.head()

,title,url,source_name,published_date,article_id,summary,sentiment_label,severity,persons,locations,organizations
0,Apocalypse Rising: We Have Reached A Moment In...,http://theeconomiccollapseblog.com/apocalypse-...,Theeconomiccollapseblog.com,2026-03-22,3c660bfc2f1ad456a114cc3cb1682ef2,Apocalypse Rising: We Have Reached A Moment In...,Negative,High,[Trump],[Iran],[Rising]
1,"Full transcript of ""Face the Nation with Marga...",https://www.cbsnews.com/news/face-the-nation-f...,CBS News,2026-03-22,7943de934befea3833290627ab3472c3,"Full transcript of ""Face the Nation with Marga...",Neutral,Medium,"[Mike Waltz, Mark Rutte, Margaret Brennan]",[],"[NATO, U.N]"
2,Transcript: NATO Secretary General Mark Rutte ...,https://www.cbsnews.com/news/mark-rutte-nato-s...,CBS News,2026-03-22,cded56559172cb24b5c01d2bd98ea045,Transcript: NATO Secretary General Mark Rutte ...,Neutral,Medium,"[Margaret Brennan, Mark Rutte]",[],[NATO]
3,"Sunday Summary: War, Oil Blockades, Interest R...",http://commercialobserver.com/2026/03/sunday-s...,Commercial Observer,2026-03-22,e5e727303ddb3e511d9f95d0a2dfff95,"Sunday Summary: War, Oil Blockades, Interest R...",Negative,High,[],"[Israel, United States, Iran]",[]
4,Bridging medical realities in the study of tec...,https://news.mit.edu/2026/bridging-medical-rea...,Mit.edu,2026-03-22,edbeb8c554de6c5b3aa19d86ed858aa8,Bridging medical realities in the study of tec...,Neutral,Medium,[Amy Moran-Thomas],[],[MIT]


## RAG Pipeline

### Step 5 — Prepare documents for embedding

Convert each article row into a structured `{page_content, metadata}` dictionary. Documents with missing content are dropped.

In [21]:
documents = article_df.apply(prepare_documents, axis = 1).dropna().tolist()

Generate dense vector embeddings for all newly fetched articles using the sentence transformer model.

In [22]:
faiss_index, metadata_store = build_embeddings(
    embedding_model = model_pipelines['embedding_model'],
    documents = documents
)

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 14.92it/s]


### Step 6: Update the vector store

Merge the new embeddings into the existing FAISS index on disk. Reload the merged index so all subsequent searches include both old and newly fetched articles.

In [23]:
# Save to vector store then reload merged index
update_vector_store(
    faiss_index,
    metadata_store,
    default_parameters["vector_store_path"],
    "faiss.index",
    "metadata.pkl"
)
faiss_index, metadata_store = load_vector_store(default_parameters["vector_store_path"])

### Step 7: Re-search the updated vector store

Now that fresh articles are merged in, perform similarity search again on the updated index. This ensures the most relevant documents including newly fetched ones are retrieved.

In [24]:
# Search again on merged index
_, nearest_k_indices = similarity_search(
    faiss_index,
    user_query_vector,
    k = default_parameters["max_top_k_results"]
)
filtered_indices = filter_by_metadata(
    indices = nearest_k_indices,
    metadata_store = metadata_store,
    filter_criteria = query_metadata
)

## Filtered Indices from the vector store
print(f"Filtered Indices: {filtered_indices}")

Filtered Indices: [164, 156, 179, 167, 24, 160, 175]


Build the final document list from the re-searched filtered indices. These documents form the context passed to the LLM.

In [25]:
# Build documents for LLM context
documents = [
    {"page_content": metadata_store[i].get("page_content"), "metadata": metadata_store[i]}
    for i in filtered_indices
]

**Checkpoint** - Pipeline time up to this point, excluding LLM inference.

In [26]:
print(f"Time taken yet: {datetime.datetime.now() - start}")

Time taken yet: 0:00:06.581766


### Step 8: Form the context window

Select the top N most relevant documents and truncate to a maximum token budget. This keeps the LLM prompt concise and within Mistral's context limit while preserving the most relevant information.

In [27]:
# Build Context Window
context_window = context_forming(
    n = 10,
    docs = documents,
    max_context_tokens = 2000,
    avg_chars_per_token = 5
)

### Step 9: Generate the answer

Pass the user query and context window to Mistral running locally via Ollama. Temperature is set to 0 for deterministic, factual responses.

In [28]:
# Generate LLM response
response = generate_answer(
    user_query,
    context = context_window,
    model = "mistral",
    temperature = 0
)

### Result

Final response generated by Mistral based on the retrieved and filtered news context.

In [29]:
print(f"User Query: {user_query}\n")
print(f"Response: {response}")

User Query: What's happening with the U.S. economy?

Response:  The available articles do not provide a comprehensive overview of the current state of the U.S. economy. However, two articles suggest that the Iran war and its impact on energy markets are affecting the economic and monetary policy outlook, as one Federal Reserve official called for notably more interest rate cuts than most U.S. central bankers. Additionally, rising gas prices due to the Iran war have already cost U.S. drivers an extra $4 billion. Another article mentions the potential economic threat of the H-1B visa program, but does not directly discuss the U.S. economy. The last article discusses U.S. tech firms demanding security restrictions against Chinese robots, but this does not directly relate to the U.S. economy.


Total **end-to-end** pipeline time including LLM inference.

In [30]:
print(f"Time taken to generate a response: {datetime.datetime.now() - start}")

Time taken to generate a response: 0:00:34.616268
